# Unit 5 Hands-On ①: ML-Agents — SnowballTarget

이 노트북은 **Hugging Face 딥 강화학습 강좌 Unit 5**의 두 번째 실습입니다.  

눈덩이를 발사하여 목표물을 맞히는 에이전트를 **PPO** 알고리즘으로 훈련합니다.  
이 유닛은 이전 유닛들과 달리 Python 코드가 아닌 **ML-Agents CLI + YAML 설정 파일** 기반입니다.

### 이전 유닛과의 차이점

| 항목 | Unit 1~4 | Unit 5 (ML-Agents) |
|---|---|---|
| **환경** | gymnasium / PLE | Unity 빌드 실행 파일 |
| **훈련 방법** | Python 코드 직접 작성 | `mlagents-learn` CLI 명령어 |
| **설정** | Python 딕셔너리 | YAML 파일 |
| **결과 영상** | imageio로 저장 | Hub에 업로드 후 브라우저에서 확인 |

---
## 목차
1. 저장소 클론 및 가상 환경 설정
2. ML-Agents 설치
3. Google Drive 마운트
4. SnowballTarget 환경 다운로드
5. 설정 파일(YAML) 생성
6. 에이전트 훈련
7. 훈련 결과 확인 (TensorBoard)
8. Hugging Face Hub 업로드
9. 브라우저에서 에이전트 플레이 확인


---
## 1. 저장소 클론 및 가상 환경 설정

ML-Agents는 Python 3.10.12 버전이 필요합니다.  
Colab의 기본 Python 버전이 맞지 않을 수 있으므로 Miniconda로 가상환경을 생성합니다.


In [1]:
# 현재 Colab 커널의 Python 버전 확인
!python --version

Python 3.12.13


In [2]:
# ML-Agents 소스코드 클론 (약 2~3분 소요)
!git clone --depth 1 https://github.com/Unity-Technologies/ml-agents

Cloning into 'ml-agents'...
remote: Enumerating objects: 2381, done.
remote: Counting objects: 100% (2381/2381), done.
remote: Compressing objects: 100% (1773/1773), done.
remote: Total 2381 (delta 863), reused 1638 (delta 591), pack-reused 0 (from 0)
Receiving objects: 100% (2381/2381), 97.19 MiB | 26.96 MiB/s, done.
Resolving deltas: 100% (863/863), done.
Updating files: 100% (2219/2219), done.


In [9]:
import os
import sys
import shutil
import subprocess
import urllib.request

MINICONDA_DIR = '/content/miniconda3'
ENV_NAME = 'mlagents'

if 'google.colab' in sys.modules:
    candidate_paths = [
        os.path.join(MINICONDA_DIR, 'bin', 'conda'),
        os.path.expanduser('~/miniconda3/bin/conda'),
        '/usr/local/bin/conda',
    ]
    conda_path = next((p for p in candidate_paths if os.path.exists(p)), None)

    if not conda_path:
        print('Colab 환경에서 Miniconda를 설치합니다...')
        installer = '/tmp/miniconda.sh'
        urls = [
            'https://repo.anaconda.com/miniconda/Miniconda3-py310_24.11.3-0-Linux-x86_64.sh',
            'https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh',
        ]

        downloaded = False
        for url in urls:
            try:
                print(f'다운로드 시도: {url}')
                urllib.request.urlretrieve(url, installer)
                downloaded = True
                print('✅ 다운로드 완료')
                break
            except Exception as e:
                print(f'⚠️ 실패: {url} -> {e}')

        if not downloaded:
            raise RuntimeError('Miniconda 설치 파일을 다운로드하지 못했습니다. Colab 네트워크 상태를 확인하세요.')

        result = subprocess.run(
            ['bash', installer, '-b', '-p', MINICONDA_DIR],
            capture_output=True,
            text=True,
            check=False,
        )
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        conda_path = next((p for p in candidate_paths if os.path.exists(p)), None)

    if not conda_path:
        raise RuntimeError('Miniconda 설치가 완료되지 않았습니다. Colab 새로고침 또는 새 런타임으로 다시 시도하세요.')

    print('conda 경로:', conda_path)

    # conda 약관 동의 (필수)
    print('conda 채널 약관을 승인합니다...')
    subprocess.run(
        [conda_path, 'tos', 'accept', '--override-channels', '--channel', 'https://repo.anaconda.com/pkgs/main'],
        capture_output=True,
        text=True,
        check=False,
    )
    subprocess.run(
        [conda_path, 'tos', 'accept', '--override-channels', '--channel', 'https://repo.anaconda.com/pkgs/r'],
        capture_output=True,
        text=True,
        check=False,
    )

    env_check = subprocess.run(
        [conda_path, 'env', 'list'],
        capture_output=True,
        text=True,
        check=False,
    )
    print(env_check.stdout)

    try:
        subprocess.run(
            [conda_path, 'create', '-y', '-n', ENV_NAME, 'python=3.10.12', 'ujson'],
            capture_output=True,
            text=True,
            check=True,
        )
        print('✅ conda 환경 생성 완료')
    except subprocess.CalledProcessError as e:
        print('❌ conda 환경 생성 실패')
        print('STDOUT:')
        print(e.stdout)
        print('STDERR:')
        print(e.stderr)
        if 'already exists' in (e.stdout or '') + (e.stderr or ''):
            print('ℹ️ 이미 같은 이름의 conda 환경이 존재합니다. 기존 환경을 사용합니다.')
        else:
            raise

    print('환경 이름:', ENV_NAME)
else:
    print('현재 노트북은 Colab/Linux 환경이 아닙니다.')
    if shutil.which('conda'):
        print('로컬에 conda가 이미 설치되어 있습니다.')
        print(f'아래 명령으로 환경을 생성하세요: conda create -y -n {ENV_NAME} python=3.10.12 ujson')
    else:
        print('이 노트북은 Google Colab용입니다. 로컬 Windows/VS Code에서는 Miniconda를 먼저 설치한 뒤 다시 실행하세요.')
        print('설치 링크: https://www.anaconda.com/download/success')


conda 경로: /content/miniconda3/bin/conda
conda 채널 약관을 승인합니다...

# conda environments:
#
# * -> active
# + -> frozen
base                     /content/miniconda3


✅ conda 환경 생성 완료
환경 이름: mlagents


In [10]:
import os
import subprocess

MINICONDA_DIR = '/content/miniconda3'
ENV_NAME = 'mlagents'
conda_path = os.path.join(MINICONDA_DIR, 'bin', 'conda')

if os.path.exists(conda_path):
    result = subprocess.run(
        [conda_path, 'run', '-n', ENV_NAME, 'python', '--version'],
        capture_output=True,
        text=True,
        check=False,
    )
    print(result.stdout.strip() or result.stderr.strip())
else:
    print(f'conda 실행 파일이 없습니다: {conda_path}')
    print('이전 셀에서 Miniconda 설치가 실패했는지 확인하세요.')


Python 3.10.12


---
## 2. ML-Agents 설치

`ml-agents-envs`: Unity 환경과 통신하는 Python 패키지  
`ml-agents`: 훈련 알고리즘 및 CLI 도구 (`mlagents-learn`, `mlagents-push-to-hf`)


In [12]:
# conda 환경에 ML-Agents 패키지를 설치합니다.
import os
import subprocess

%cd /content/ml-agents

commands = [
    ["/content/miniconda3/bin/conda", "run", "-n", "mlagents", "python", "-m", "pip", "install", "--upgrade", "pip"],
    ["/content/miniconda3/bin/conda", "run", "-n", "mlagents", "python", "-m", "pip", "install", "-e", "./ml-agents-envs"],
    ["/content/miniconda3/bin/conda", "run", "-n", "mlagents", "python", "-m", "pip", "install", "-e", "./ml-agents"],
]

for cmd in commands:
    print(f"\n>>> {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"명령 실패: {' '.join(cmd)}")

print('✅ ML-Agents 설치 완료')


/content/ml-agents

>>> /content/miniconda3/bin/conda run -n mlagents python -m pip install --upgrade pip


>>> /content/miniconda3/bin/conda run -n mlagents python -m pip install -e ./ml-agents-envs
Obtaining file:///content/ml-agents/ml-agents-envs
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for mlagents_envs (pyproject.toml): started
  Building editable for mlagents_envs (pyproject.toml): finished with status 'done'
  Created wheel for mlagents_envs: filename=mlagents_envs-1.2.0.dev0-0.editable-py3-none-any.whl size=393

In [ ]:
# 설치 확인: 버전이 출력되면 정상
!{MINICONDA_DIR}/bin/conda run -n {ENV_NAME} mlagents-learn --help | head -5

---
## 3. Google Drive 마운트

Colab VM은 세션 종료 시 파일이 모두 삭제됩니다.  
훈련 결과(`results/`)를 Drive에 심볼릭 링크로 연결하여 영구 보존합니다.

```
Google Drive/RL_Course/Unit5_SnowballTarget/
└── results/
    └── SnowballTarget1/   ← 훈련 체크포인트 및 최종 모델(.onnx)
```


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ✏️ 저장 경로를 원하는 대로 변경하세요.
DRIVE_BASE   = '/content/drive/MyDrive/RL_Course/Unit5_SnowballTarget'
DRIVE_RESULTS = f'{DRIVE_BASE}/results'

os.makedirs(DRIVE_RESULTS, exist_ok=True)

# ML-Agents의 결과 폴더를 Drive 경로로 심볼릭 링크
# 훈련 중 생성되는 모든 파일이 Drive에 직접 저장됨
if not os.path.exists('/content/ml-agents/results'):
    os.symlink(DRIVE_RESULTS, '/content/ml-agents/results')
    print(f'✅ 심볼릭 링크 생성: /content/ml-agents/results → {DRIVE_RESULTS}')
else:
    print('ℹ️  results 폴더가 이미 존재합니다.')

print(f'결과 저장 경로: {DRIVE_RESULTS}')


---
## 4. SnowballTarget 환경 다운로드

ML-Agents는 Unity로 빌드된 **실행 파일(.x86_64)**을 환경으로 사용합니다.  
Colab은 Linux(Ubuntu) 환경이므로 Linux용 실행 파일을 다운로드합니다.


In [ ]:
# 환경 실행 파일 저장 디렉토리 생성
!mkdir -p ./training-envs-executables/linux

In [ ]:
# SnowballTarget 환경 다운로드
!wget -q "https://github.com/huggingface/Snowball-Target/raw/main/SnowballTarget.zip" \
    -O ./training-envs-executables/linux/SnowballTarget.zip
print('✅ 다운로드 완료')

In [ ]:
# 압축 해제
!unzip -q -d ./training-envs-executables/linux/ \
    ./training-envs-executables/linux/SnowballTarget.zip
print('✅ 압축 해제 완료')

In [ ]:
# 실행 권한 부여 (없으면 mlagents-learn 실행 시 Permission denied 오류)
!chmod -R 755 ./training-envs-executables/linux/SnowballTarget
print('✅ 실행 권한 설정 완료')

---
## 5. 설정 파일(YAML) 생성

ML-Agents의 훈련 하이퍼파라미터는 YAML 파일로 정의합니다.

| 파라미터 | 값 | 설명 |
|---|---|---|
| `trainer_type` | ppo | 알고리즘: Proximal Policy Optimization |
| `max_steps` | 200,000 | 총 훈련 스텝 |
| `learning_rate` | 0.0003 | 학습률 |
| `batch_size` | 128 | 미니배치 크기 |
| `buffer_size` | 2048 | 경험 버퍼 크기 |
| `beta` | 0.005 | 엔트로피 계수 (탐색 장려) |
| `epsilon` | 0.2 | PPO 클리핑 범위 |
| `gamma` | 0.99 | 할인율 |
| `hidden_units` | 256 | 신경망 은닉층 크기 |
| `num_layers` | 2 | 신경망 레이어 수 |
| `checkpoint_interval` | 50,000 | 체크포인트 저장 주기 |

> ✏️ 하이퍼파라미터를 수정한 뒤 훈련 결과를 비교해보세요.  
> 각 파라미터 설명: https://github.com/Unity-Technologies/ml-agents/blob/main/docs/Training-Configuration-File.md


In [ ]:
# SnowballTarget.yaml 설정 파일 생성(GPU 학습용)
snowball_config = """\
behaviors:
  SnowballTarget:
    trainer_type: ppo
    summary_freq: 10000
    keep_checkpoints: 10
    checkpoint_interval: 50000
    max_steps: 200000
    time_horizon: 64
    threaded: true
    hyperparameters:
      learning_rate: 0.0003
      learning_rate_schedule: linear
      batch_size: 128
      buffer_size: 2048
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
    network_settings:
      normalize: false
      hidden_units: 256
      num_layers: 2
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.99
        strength: 1.0
"""

config_path = './config/ppo/SnowballTarget.yaml'
with open(config_path, 'w') as f:
    f.write(snowball_config)

print(f'✅ 설정 파일 생성: {config_path}')
print(snowball_config)


---
## 6. 에이전트 훈련

```
mlagents-learn <설정파일>                    ← YAML 하이퍼파라미터
    --env=<환경실행파일>                      ← Unity 빌드 실행 파일 경로
    --run-id=<실행ID>                        ← 훈련 결과 식별자 (폴더명)
    --no-graphics                            ← 렌더링 없이 실행 (Colab 필수)
    --resume                                 ← 중단된 훈련 이어서 재개
```

> ⏱️ 예상 훈련 시간: **10~35분** (GPU 사용 시)  
> 첫 실행에서 `--resume` 플래그로 오류가 나면, 해당 플래그 없이 재실행하세요.


In [23]:
# SnowballTarget 에이전트 훈련
# Colab 커널이 아닌 conda 환경에서 실행합니다.
!{MINICONDA_DIR}/bin/conda run -n {ENV_NAME} mlagents-learn ./config/ppo/SnowballTarget.yaml \
    --env=./training-envs-executables/linux/SnowballTarget/SnowballTarget \
    --run-id="SnowballTarget1" \
    --no-graphics \
    --resume


            ┐  ╖
        ╓╖╬│╡  ││╬╖╖
    ╓╖╬│││││┘  ╬│││││╬╖
 ╖╬│││││╬╜        ╙╬│││││╖╖                               ╗╗╗
 ╬╬╬╬╖││╦╖        ╖╬││╗╣╣╣╬      ╟╣╣╬    ╟╣╣╣             ╜╜╜  ╟╣╣
 ╬╬╬╬╬╬╬╬╖│╬╖╖╓╬╪│╓╣╣╣╣╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╒╣╣╖╗╣╣╣╗   ╣╣╣ ╣╣╣╣╣╣ ╟╣╣╖   ╣╣╣
 ╬╬╬╬┐  ╙╬╬╬╬│╓╣╣╣╝╜  ╫╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╟╣╣╣╙ ╙╣╣╣  ╣╣╣ ╙╟╣╣╜╙  ╫╣╣  ╟╣╣
 ╬╬╬╬┐     ╙╬╬╣╣      ╫╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╟╣╣╬   ╣╣╣  ╣╣╣  ╟╣╣     ╣╣╣┌╣╣╜
 ╬╬╬╜       ╬╬╣╣      ╙╝╣╣╬      ╙╣╣╣╗╖╓╗╣╣╣╜ ╟╣╣╬   ╣╣╣  ╣╣╣  ╟╣╣╦╓    ╣╣╣╣╣
 ╙   ╓╦╖    ╬╬╣╣   ╓╗╗╖            ╙╝╣╣╣╣╝╜   ╘╝╝╜   ╝╝╝  ╝╝╝   ╙╣╣╣    ╟╣╣╣
   ╩╬╬╬╬╬╬╦╦╬╬╣╣╗╣╣╣╣╣╣╣╝                                             ╫╣╣╣╣
      ╙╬╬╬╬╬╬╬╣╣╣╣╣╣╝╜
          ╙╬╬╬╣╣╣╜
             ╙
        
 Version information:
  ml-agents: 1.2.0.dev0,
  ml-agents-envs: 1.2.0.dev0,
  Communicator API: 1.5.0,
  PyTorch: 2.8.0+cu128
[INFO] Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0
[INFO] Connected new brain: SnowballTarget?

---
## 7. 훈련 결과 확인 (TensorBoard)

ML-Agents는 훈련 중 자동으로 TensorBoard 로그를 생성합니다.  
아래 셀을 실행하면 노트북 안에서 바로 그래프를 확인할 수 있습니다.

**주요 지표:**
- `Environment/Cumulative Reward`: 에피소드 누적 보상 (높을수록 좋음)
- `Environment/Episode Length`: 에피소드 길이
- `Policy/Entropy`: 탐색 엔트로피 (훈련이 진행될수록 감소)
- `Losses/Policy Loss`: 정책 손실

> 성공 기준: **Mean Reward ≥ 15** (에피소드당 15개 이상 목표물 명중)


In [24]:
# TensorBoard 실행 (훈련 중 또는 훈련 후 실행 가능)
%load_ext tensorboard
%tensorboard --logdir results/SnowballTarget1

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


<IPython.core.display.Javascript object>

In [28]:
# 결과 파일 확인
!ls results/SnowballTarget1/SnowballTarget/

checkpoint.pt					    SnowballTarget-39984.onnx
events.out.tfevents.1785708370.5cc2e09ca4d1.4487.0  SnowballTarget-39984.pt
events.out.tfevents.1785708588.5cc2e09ca4d1.5453.0  SnowballTarget-50584.onnx
SnowballTarget-19992.onnx			    SnowballTarget-50584.pt
SnowballTarget-19992.pt


---
## 8. Hugging Face Hub 업로드

`mlagents-push-to-hf` 명령어가 아래 작업을 자동으로 처리합니다:
- 훈련된 모델(`.onnx`) 업로드
- 모델 카드 자동 생성
- TensorBoard 로그 업로드

### 파라미터
| 파라미터 | 설명 | 예시 |
|---|---|---|
| `--run-id` | 훈련 실행 ID | `SnowballTarget1` |
| `--local-dir` | 로컬 결과 폴더 | `./results/SnowballTarget1` |
| `--repo-id` | HF Hub 저장소 | `username/ppo-SnowballTarget` |
| `--commit-message` | 커밋 메시지 | `"First Push"` |

### 사전 준비
1. [HF 계정 생성](https://huggingface.co/join)
2. [Write 토큰 발급](https://huggingface.co/settings/tokens)


In [ ]:
from huggingface_hub import login

# ✏️ 본인의 HF 토큰 입력
login(token='hf_xxxxxxxxxxxxxxxxxxxxxxxx')


In [30]:
# ✏️ 아래 값을 본인 정보로 수정하세요.
RUN_ID     = 'SnowballTarget1'
LOCAL_DIR  = f'./results/{RUN_ID}'
REPO_ID    = 'DitDahDitDit/ppo-SnowballTarget'  # ← username 변경
COMMIT_MSG = 'First Push'

!{MINICONDA_DIR}/bin/conda run -n {ENV_NAME} mlagents-push-to-hf \
    --run-id={RUN_ID} \
    --local-dir={LOCAL_DIR} \
    --repo-id={REPO_ID} \
    --commit-message="{COMMIT_MSG}"


[INFO] This function will create a model card and upload your SnowballTarget1 into HuggingFace Hub. This is a work in progress: If you encounter a bug, please send open an issue
[INFO] Pushing repo SnowballTarget1 to the Hugging Face Hub
[INFO] Your model is pushed to the hub. You can view your model here: https://huggingface.co/DitDahDitDit/ppo-SnowballTarget


---
## 9. 브라우저에서 에이전트 플레이 확인

업로드 완료 후 아래 HF Space에서 에이전트가 플레이하는 모습을 실시간으로 확인할 수 있습니다.

1. 아래 링크 접속: https://huggingface.co/spaces/ThomasSimonini/ML-Agents-SnowballTarget
2. **Step 1**: 본인의 HF username 입력 → 검색
3. **Step 2**: 모델 저장소 선택
4. **Step 3**: `SnowballTarget.onnx` 선택 (가장 최신 모델)

> 💡 체크포인트별로 다른 모델(`SnowballTarget-50000.onnx`, `SnowballTarget-100000.onnx` 등)을  
> 선택해서 훈련 단계별 성능 변화를 비교해볼 수 있습니다.

> 🎯 목표 점수: **Mean Reward ≥ 15** (에피소드당 15개 이상 목표물 명중)


In [31]:
# 업로드된 모델 링크 출력
print(f'모델 확인: https://huggingface.co/{REPO_ID}')
print(f'플레이 확인: https://huggingface.co/spaces/ThomasSimonini/ML-Agents-SnowballTarget')


모델 확인: https://huggingface.co/DitDahDitDit/ppo-SnowballTarget
플레이 확인: https://huggingface.co/spaces/ThomasSimonini/ML-Agents-SnowballTarget
